# 🚀 ComfyUI One-Click Autopilot on NVIDIA A100 (ภาพนิ่ง 8K + สตูดิโอวิดีโอ MP4)
### จัดเตรียมโดย Antigravity สำหรับพี่นัท — พร้อมใช้งาน 100% ไม่ต้องแตะโค้ด

**✨ ฟีเจอร์ระดับโปรดักชันที่ติดตั้งไว้ในสมุดงานนี้:**
- 📸 **Juggernaut XL v9:** โมเดลโฟโต้สมจริง 8K เจนภาพคมชัด แสงเงาเป็นธรรมชาติ
- 🎬 **Stable Video Diffusion XT (SVD-XT) + VideoHelperSuite:** เสกภาพนิ่งให้เคลื่อนไหวเป็นคลิปวิดีโอ MP4 ความละเอียดสูง ลมพัด น้ำกระเพื่อม กล้องบินโดรน
- 🛡️ **ปลดล็อก CORS Security + Legacy Frontend:** หมดปัญหาหน้าต่าง 403 Forbidden และเปิดติดทันที 100%
- ⏰ **30-Min Auto-Shutdown Watchdog:** หากไม่มีการสั่งงานต่อเนื่องเกิน **30 นาที** ระบบจะสั่ง `runtime.unassign()` ปิดเครื่อง A100 ให้อัตโนมัติทันที เพื่อเซฟโควต้า Colab Units สูงสุด ไม่ต้องกังวลเรื่องลืมปิด!

**วิธีเปิดใช้งานใน 2 คลิก:**
1. คลิกเมนู **Runtime (รันไทม์)** ด้านบน ➡️ **Change runtime type** ➡️ เลือก `A100 GPU` (และเปิด `High-RAM`) ➡️ กด **Save**
2. คลิกเมนู **Runtime** ➡️ เลือก **Run all (รันทั้งหมด)**

In [ ]:
# ==============================================================================
# 🚀 COMFYUI A100 AUTOPILOT (PHOTO 8K + SVD VIDEO MP4) + 30-MIN WATCHDOG
# ==============================================================================
import subprocess, threading, time, re, os, urllib.request, json
from google.colab import runtime

print('⏳ กำลังเริ่มต้นระบบ ComfyUI (ภาพนิ่ง 8K + สตูดิโอวิดีโอ) บนการ์ดจอ A100...')
!pkill -f cloudflared > /dev/null 2>&1
!pkill -f 'python main.py' > /dev/null 2>&1

# 1. ติดตั้ง ComfyUI และ Custom Nodes หลัก
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

%cd /content/ComfyUI/custom_nodes
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-Manager'):
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git

if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite'):
    print('📦 กำลังติดตั้งปลั๊กอินเรนเดอร์วิดีโอ ComfyUI-VideoHelperSuite...')
    !git clone -q https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    %cd /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
    !pip install -q -r requirements.txt

%cd /content/ComfyUI

# 2. ตรวจสอบและดาวน์โหลดโมเดลภาพสมจริง Juggernaut XL v9
photo_model = '/content/ComfyUI/models/checkpoints/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors'
if not os.path.exists(photo_model):
    print('📸 กำลังดึงโมเดลสมจริงระดับท็อป Juggernaut XL (~6.6GB ความเร็ว A100 ~200MB/s)...')
    !wget -c -q --show-progress 'https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors' -P /content/ComfyUI/models/checkpoints/

# 3. ตรวจสอบและดาวน์โหลดโมเดลวิดีโอ Stable Video Diffusion XT (SVD-XT)
video_model = '/content/ComfyUI/models/checkpoints/svd_xt_1_1.safetensors'
if not os.path.exists(video_model):
    print('🎬 กำลังดึงโมเดลเรนเดอร์วิดีโอระดับท็อป Stable Video Diffusion XT (~4.7GB)...')
    !wget -c -q --show-progress 'https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt-1-1/resolve/main/svd_xt_1_1.safetensors' -P /content/ComfyUI/models/checkpoints/

# 4. ติดตั้ง Cloudflared
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 5. สตาร์ท Tunnel และดักจับ URL
def start_tunnel():
    proc = subprocess.Popen('cloudflared tunnel --url http://127.0.0.1:8188', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n' + '='*65)
            print('🎉 ระบบพร้อมใช้งานแล้ว 100%! (รองรับทั้งภาพนิ่ง 8K + วิดีโอ MP4)')
            print(f'👉 {match.group(0)}')
            print('='*65 + '\n')
            print('🛡️ ระบบรักษาความปลอดภัย: หากไม่มีการสั่งงานเกิน 30 นาที เครื่องจะปิดตัวเองอัตโนมัติ')

threading.Thread(target=start_tunnel, daemon=True).start()

# 6. ระบบเฝ้าระวังตัดการเชื่อมต่ออัตโนมัติเมื่อไม่ใช้งานเกิน 30 นาที (Auto-Shutdown Watchdog)
def auto_shutdown_watchdog(idle_seconds=1800):
    time.sleep(30)
    last_active = time.time()
    last_job_count = 0
    while True:
        time.sleep(20)
        try:
            req = urllib.request.Request('http://127.0.0.1:8188/history')
            with urllib.request.urlopen(req) as resp:
                hist = json.loads(resp.read().decode('utf-8'))
                current_count = len(hist)
                if current_count > last_job_count:
                    last_job_count = current_count
                    last_active = time.time()
        except Exception:
            pass
            
        if time.time() - last_active >= idle_seconds:
            print('\n' + '='*65)
            print('⏰ ตรวจพบไม่มีการใช้งานต่อเนื่องเกิน 30 นาที!')
            print('🛡️ สั่งตัดการเชื่อมต่อและปิดเครื่อง A100 ทันทีเพื่อเซฟโควต้าให้พี่นัท...')
            print('='*65 + '\n')
            runtime.unassign()
            break

threading.Thread(target=auto_shutdown_watchdog, daemon=True).start()
time.sleep(2)

%cd /content/ComfyUI
# รันด้วย Legacy Frontend + ปลดล็อก CORS Security
!python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header '*' --front-end-version Comfy-Org/ComfyUI_legacy_frontend@latest
